In [4]:
import os
import pandas as pd
import pyodbc
from datetime import datetime
import warnings

# Configurações iniciais
usuario = os.getenv('USERNAME')
warnings.filterwarnings('ignore')

# Caminho e leitura das credenciais
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o banco
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Consulta SQL resumida por cliente
query = """
SELECT 
    DP.CD_InscricaoNacional,
    COUNT(DISTINCT FT.ID_Cota) AS Qtd_Cotas,
    SUM(FT.VL_Bem) AS Soma_VL_Bem,
    AVG(FT.VL_Bem) AS Media_VL_Bem
FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP
    ON FT.ID_Pessoa = DP.ID_Pessoa
WHERE
    FT.ST_Contrato = 'Ativo'
    AND FT.Tipo_Pessoa = 'F'
    AND DP.CD_InscricaoNacional IS NOT NULL
GROUP BY 
    DP.CD_InscricaoNacional
"""

# Executa a query
print("📥 Executando a consulta SQL...")
df = pd.read_sql(query, conn)

# Cálculo do total geral para a diretoria
total_clientes = df['CD_InscricaoNacional'].nunique()
total_cotas = df['Qtd_Cotas'].sum()
soma_total = df['Soma_VL_Bem'].sum()
media_geral = df['Media_VL_Bem'].mean()

df_resumo = pd.DataFrame({
    'Indicador': ['Clientes Únicos', 'Total de Cotas', 'Soma VL_Bem', 'Média VL_Bem por Cliente'],
    'Valor': [total_clientes, total_cotas, soma_total, media_geral]
})

# Caminho com data no nome
data_hoje = datetime.today().strftime('%Y-%m-%d')
output_path = fr"C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\DADOS MESA DE DECISÃO\DADOS VALOR DO BEM POR CLIENTE\resumo_valor_bem_{data_hoje}.xlsx"

# Salvando em duas abas: detalhado por cliente e resumo executivo
with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='Detalhado_por_Cliente', index=False)
    df_resumo.to_excel(writer, sheet_name='Resumo_Executivo', index=False)

print(f"✅ Arquivo salvo com sucesso em:\n{output_path}")


📥 Executando a consulta SQL...
✅ Arquivo salvo com sucesso em:
C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\DADOS MESA DE DECISÃO\DADOS VALOR DO BEM POR CLIENTE\resumo_valor_bem_2025-05-09.xlsx
